# Solaris RL — Kaggle Training Notebook

Resumable DQN / Double DQN training for Atari Solaris.

**Kaggle session limit: 9 hours, hard cutoff.** This notebook is designed to be
re-run across multiple sessions — it auto-resumes from the last saved checkpoint
each time, using Kaggle's persistent **Output** directory to carry checkpoints
between sessions.

### How to use this notebook across multiple sessions
1. **First run:** Settings (right sidebar) → Accelerator → **GPU T4 x2**. Run all cells.
2. When the 9-hour limit hits (or you stop it manually), the notebook stops —
 but checkpoints are already saved in `/kaggle/working/checkpoints/`.
3. **To continue:** open this same notebook again (it auto-saves your edits),
 click **Save Version → Save & Run All**. Kaggle restores your previous
 `/kaggle/working/` output automatically, so checkpoints persist and
 training resumes exactly where it left off.
4. Repeat until `total_steps` is reached.

### Before you start
- Push your `atari-solaris` project to a **public (or Kaggle-accessible private) GitHub repo** first.
- Set `REPO_URL` in the cell below.
- Metrics are written to TensorBoard under `checkpoints/<exp_name>/tb_logs/`.
 Download that folder after the run and view with `tensorboard --logdir tb_logs`.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# ============================================================
# CONFIG — edit these before running
# ============================================================
REPO_URL = "https://github.com/3rdDerivativeofPi/atari-solaris.git" # <-- EDIT THIS
BRANCH = "main"

# Which experiment config to run. Must match a file in configs/.
CONFIG_PATH = "configs/double_dqn.yaml"
SEED = 616

In [ ]:
# ============================================================
# 1. Clone (or update) the repo into /kaggle/working
# ============================================================
# /kaggle/working persists across sessions for the SAME notebook, so we clone
# once and `git pull` on subsequent runs rather than re-cloning every time.
import os

PROJECT_DIR = "/kaggle/working/atari-solaris"

if not os.path.isdir(PROJECT_DIR):
    print("Cloning repo (first run)...")
    !git clone --branch {BRANCH} {REPO_URL} {PROJECT_DIR}
else:
    print("Repo already present — pulling latest changes...")
    !cd {PROJECT_DIR} && git pull

os.chdir(PROJECT_DIR)
print("\nWorking directory:", os.getcwd())

In [ ]:
# ============================================================
# 2. Install dependencies
# ============================================================
# Kaggle images already ship PyTorch + CUDA, so we skip reinstalling torch
# (reinstalling it tends to break the preconfigured CUDA linkage).
!pip install -q ale-py gymnasium[accept-rom-license] AutoROM opencv-python-headless tensorboard pyyaml
!AutoROM --accept-license -y > /dev/null 2>&1
print("OK Dependencies installed")

In [ ]:
# ============================================================
# 3. Verify GPU and environment
# ============================================================
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

import gymnasium as gym
import ale_py
gym.register_envs(ale_py)
env = gym.make("ALE/Solaris-v5", frameskip=1)
obs, _ = env.reset()
print("✅ Solaris env OK — obs shape:", obs.shape)
env.close()

In [ ]:
# ============================================================
# 4. Set up TensorBoard log directory (local-only metrics)
# ============================================================
# Metrics are written to /kaggle/working/checkpoints/<exp_name>/tb_logs/
# which is part of Kaggle's persistent Output. Download that folder after
# the run and view locally with:
# tensorboard --logdir tb_logs
import os
TB_ROOT = os.path.join(PROJECT_DIR, "checkpoints")
os.makedirs(TB_ROOT, exist_ok=True)
print("TensorBoard logs will be written under:", TB_ROOT)

In [ ]:
# ============================================================
# 5. Symlink checkpoints into /kaggle/working so they PERSIST
#    across sessions (this is the critical resumability step)
# ============================================================
# Everything under /kaggle/working is saved as this notebook's "Output" and
# restored automatically the next time you open and run this same notebook.
# checkpoints/ already lives inside PROJECT_DIR, which is itself inside
# /kaggle/working, so no extra symlinking is actually needed — but we assert
# it here explicitly so it's obvious and verifiable.
ckpt_root = os.path.join(PROJECT_DIR, "checkpoints")
os.makedirs(ckpt_root, exist_ok=True)
assert ckpt_root.startswith("/kaggle/working"), "Checkpoints must live under /kaggle/working to persist!"
print("✅ Checkpoints will persist at:", ckpt_root)

# Show what's already there (non-empty on a resumed session)
for root, dirs, files in os.walk(ckpt_root):
    for f in files:
        print(" -", os.path.join(root, f))

In [ ]:
# ============================================================
# 6. Train — auto-resumes if a checkpoint already exists
# ============================================================
# --resume is always passed:
# - No checkpoint found -> train.py starts fresh
# - Checkpoint found -> weights/optimiser/step restored
!python train.py --config {CONFIG_PATH} --seed {SEED} --resume

---
### What happens when this session times out mid-training

Nothing bad — that's the point. The training loop checkpoints every
`checkpoint_freq` steps (100K by default) to `checkpoints/<exp_name>/`.
When you come back and re-run this notebook:

- The repo is already cloned (just `git pull`s for any code updates).
- `train.py --resume` finds the latest `step_*.pt` checkpoint and restores
 network weights, optimiser state, and the step counter.
- The replay buffer starts empty and refills over the first `learning_starts`
 steps — a small, expected cost, not a bug.
- TensorBoard logs under `checkpoints/<exp_name>/tb_logs/` accumulate
 across sessions, so you get one continuous learning curve.

### Estimating sessions needed
At ~150–300 SPS on a Kaggle T4 (variable with load), a 10M-step run needs
roughly **10–18 hours of GPU time total** → **2–3 Kaggle sessions** at the
9-hour cap. Kaggle gives 30 free GPU-hours/week, which covers this easily.